## Lab6-Assignment: Topic Classification

Use the same training, development, and test partitions of the the 20 newsgroups text dataset as in Lab6.4-Topic-classification-BERT.ipynb 

* Fine-tune and examine the performance of another transformer-based pretrained language models, e.g., RoBERTa, XLNet

* Compare the performance of this model to the results achieved in Lab6.4-Topic-classification-BERT.ipynb and to a conventional machine learning approach (e.g., SVM, Naive Bayes) using bag-of-words or other engineered features of your choice. 
Describe the differences in performance in terms of Precision, Recall, and F1-score evaluation metrics.

In [ ]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.metrics import classification_report
from simpletransformers.classification import ClassificationModel, ClassificationArgs
import matplotlib.pyplot as plt
import seaborn as sn

In [ ]:
from sklearn.datasets import fetch_20newsgroups

# load only a sub-selection of the categories (4 in our case)
categories = ['alt.atheism', 'comp.graphics', 'sci.med', 'sci.space']

# remove the headers, footers and quotes (to avoid overfitting)
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'), categories=categories, random_state=42)
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'), categories=categories, random_state=42)

In [ ]:
train = pd.DataFrame({'text': newsgroups_train.data, 'labels': newsgroups_train.target})
print(len(train))
train.head(5)

In [ ]:
test = pd.DataFrame({'text': newsgroups_test.data, 'labels': newsgroups_test.target})
print(len(test))
test.head(5)

In [ ]:
from sklearn.model_selection import train_test_split

train, dev = train_test_split(train, test_size=0.1, random_state=0,
                               stratify=train[['labels']])

In [ ]:
# Model configuration # https://simpletransformers.ai/docs/usage/#configuring-a-simple-transformers-model
model_args = ClassificationArgs()

model_args.overwrite_output_dir=True # overwrite existing saved models in the same directory
model_args.evaluate_during_training=True # to perform evaluation while training the model
# (eval data should be passed to the training method)

model_args.num_train_epochs=10 # number of epochs
model_args.train_batch_size=32 # batch size
model_args.learning_rate=4e-6 # learning rate
model_args.max_seq_length=256 # maximum sequence length
# Note! Increasing max_seq_len may provide better performance, but training time will increase.
# For educational purposes, we set max_seq_len to 256.

# Early stopping to combat overfitting: https://simpletransformers.ai/docs/tips-and-tricks/#using-early-stopping
model_args.use_early_stopping=True
model_args.early_stopping_delta=0.01 # "The improvement over best_eval_loss necessary to count as a better checkpoint"
model_args.early_stopping_metric='eval_loss'
model_args.early_stopping_metric_minimize=True
model_args.early_stopping_patience=2
model_args.evaluate_during_training_steps=32 # how often you want to run validation in terms of training steps (or batches)

In [ ]:
model = ClassificationModel('roberta', 'roberta-base', num_labels=4, args=model_args, use_cuda=True)

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
_, history = model.train_model(train, eval_df=dev)

In [ ]:
# Training and evaluation loss
train_loss = history['train_loss']
eval_loss = history['eval_loss']
plt.plot(train_loss, label='Training loss')
plt.plot(eval_loss, label='Evaluation loss')
plt.title('Training and evaluation loss')
plt.legend()

In [ ]:
# Evaluate the model
result, model_outputs, wrong_predictions = model.eval_model(dev)
result

In [ ]:
predicted, probabilities = model.predict(test.text.to_list())
test['predicted'] = predicted

In [ ]:
test.head(5)

In [ ]:
print(classification_report(test['labels'], test['predicted']))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

svm_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        max_features=10000,
        ngram_range=(1, 2)
    )),
    ('svm', LinearSVC())
])

svm_model.fit(train['text'], train['labels'])

svm_predicted = svm_model.predict(test['text'])

test['svm_predicted'] = svm_predicted

print(classification_report(test['labels'], test['svm_predicted']))

**Analysis**

The results of the Lab6.4-Topic-classification-BERT are as follows:

             precision    recall  f1-score   support

           0       0.84      0.81      0.83       319
           1       0.89      0.93      0.91       389
           2       0.92      0.88      0.90       396
           3       0.83      0.86      0.84       394

    accuracy                           0.87      1498
   macro avg       0.87      0.87      0.87      1498
weighted avg       0.87      0.87      0.87      1498 


Overall, we see that the BERT Model used in Lab 6.4 performs somewhat well, with precision, recall, and f1-scores all above 80. Additionally, the data set seems to be more or less balanced as the support of each class is all in the 300s. Next, the macro and weighted average scores are all good as well (in addition to the accuracy score).



Looking at the RoBERTa model we trained in this lab, we see the following results:


            precision    recall  f1-score   support

           0       0.83      0.81      0.82       319
           1       0.90      0.91      0.91       389
           2       0.91      0.87      0.89       396
           3       0.80      0.85      0.83       394

    accuracy                           0.86      1498
   macro avg       0.86      0.86      0.86      1498
weighted avg       0.86      0.86      0.86      1498


The roBERTa model has roughly similar precision, recall, and f1-scores when compared to the respective classes of the BERT model (although they generally seem to be a little worse). The macro and weighted scores are also consistently worse (though very similar) to the BERT model. This is interesting because the roBERTa model is more robust compared to the BERT model.


For our second model, we use a Support Vector Machine. The results are as follow:

            precision    recall  f1-score   support

           0       0.84      0.78      0.81       319
           1       0.87      0.88      0.88       389
           2       0.88      0.82      0.85       396
           3       0.78      0.86      0.82       394

    accuracy                           0.84      1498
   macro avg       0.84      0.84      0.84      1498
weighted avg       0.84      0.84      0.84      1498


This model performs categorically worse than BERT and roBERTa. This is best exemplified by the macro and weighted average scores of 0.84 for SVM compared to roBERTa’s 0.86 and BERT’s 0.87. This stark difference between SVM and BERT and roBERTa may be because SVM uses a TF*IDF approach to classification tasks, whereas BERT and roBERTa use word embedding which can help convey context due to their relation to Distributional Hypothesis. 

Although the performance of different models is dependent on various hyperparameters as well as the exact data, it appears that the BERT model is best for this topic classification task in this lab assignment. 
